# Establishing Connections to Databases

In [1]:
bridge_sfn = '0130192'

## TIMS 'doticsqlp31' Data Source

In [2]:
import pandas as pd
import sqlalchemy as db
from sqlalchemy import create_engine, MetaData, text, inspect
from sqlalchemy.engine import URL

In [5]:
connection_string = (
    r"Driver=ODBC Driver 17 for SQL Server;"
    r"Server=doticsqlp31;"
    r"Database=TIMS;"
    r"Trusted_Connection=yes;"
)

connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})

engine = create_engine(connection_url)
conn = engine.connect()

## 'DOTWarehousePrd' Data Source

In [4]:
connection_string = (
    r"Driver=SQL Server;"
    r"Server=DOTWarehousePrd;"
    r"Database=Warehouse;"
    r"Trusted_Connection=yes;"
)
connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
engine = create_engine(connection_url)
conn = engine.connect()

In [5]:
# Get data from 1985-1990 table - Huge, don't do this.
# hist85_90DataQuery = text("SELECT * from BMS_1985_1990")

In [6]:
# hist_bridge_data = pd.read_sql(hist85_90DataQuery, conn)
# hist_bridge_data

## Oracle Load Rating Database

In [7]:
from sqlalchemy import create_engine, text
import json

In [8]:
def load_secrets(file_path):
    with open(file_path, 'r') as file:
        user_info = json.load(file)
    return user_info

In [9]:
from pathlib import Path

file_path = Path.home() / 'secrets.json'
dane = load_secrets(file_path)

In [10]:
# Oracle connection string using service name instead of SID
oracle_connection_string = (
    f"oracle+cx_oracle://{dane['BRR_USN']}:{dane['BRR_PASS']}@{dane['BRR_SERVER']}:{dane['BRR_PORT']}/?service_name={dane['BRR_SERVICE']}"
)

In [11]:
# Create the engine
oracle_engine = create_engine(oracle_connection_string)

In [12]:
# Establish connection
oracle_conn = oracle_engine.connect()
print("Connection successful!")

Connection successful!


In [13]:
from sqlalchemy import inspect

# Create an inspector from the engine
inspector = inspect(oracle_engine)

# Get a list of table names
tables = inspector.get_table_names()
print("Tables in the database:")
for table in tables:
    print(table)

Tables in the database:


In [14]:
query = text("""
  SELECT DISTINCT owner
  FROM all_tables
  ORDER BY owner
""")

result = oracle_conn.execute(query)
schemas = [row[0] for row in result]
print("Accessible schemas:")

for schema in schemas:
     print(schema)

Accessible schemas:
BRIDGEWARE
CTXSYS
MDSYS
ODOTREF
SYS
SYSTEM
XDB


In [15]:
schema_list = ["BRIDGEWARE", "CTXSYS", "MDSYS", "ODOTREF", "SYS", "SYSTEM", 
               "XDB"]  # Add other schemas if you want to search deeper

for schema_name in schema_list:
    print(f"\nChecking for tables in schema: {schema_name}\n")
    query = text(f"""
    SELECT table_name
    FROM all_tables
    WHERE owner = '{schema_name}'
    """)
    result = oracle_conn.execute(query)
    tables = [row[0] for row in result]
    if tables:
        print(f"Tables in schema {schema_name}:")
        for table in tables:
            print(table)
    else:
        print(f"No tables found in schema {schema_name}.")


Checking for tables in schema: BRIDGEWARE

Tables in schema BRIDGEWARE:
ABW_BRIDGE_SUB_LRFDDSN_SETTING
ABW_MA_STRUSS_ELEM_LOSS_RANGE
ABW_MCB_SEG_DECK
ABW_MATL_CONC
ABW_MATL_PS_BAR
ABW_MATL_PS_STRAND
ABW_FL_FLOORBEAM_TRAVELWAY
ABW_FLINE_MBR
ABW_MCB_TEND_PROF_DEF_DETAIL
ABW_STL_ANGLE
ABW_STL_ANGLE_CONN
ABW_STL_BEAM_ASSEMBLY
ABW_WEB_DISTRIB_LOAD
BRIDGE
COPTIONS
MULTIMEDIA
ABW_TRUSS_LINE_MBR
ABW_TRUSS_LINE_SUPPORT
ABW_TRUSSDEF_ELEMENT_CONC_LOAD
ABW_RESULTS_CONC_LS_SUMMARY
ABW_RESULTS_CONC_XSEC_PROP
ABW_LIB_MATL_TIMBER_SAWN_ITEM
ABW_SYS_LRFD_LS
ABW_SUBSDEF_MODEL_SETTING
ABW_SUPER_BR_FORCE_SUB
ABW_CONC_CURB_SIDEWALK
ABW_CONC_RAILING
ABW_CONC_RAILING_LOC
ABW_BRIDGE_DIAPHRAGM_DEF_CON
ABW_SUBSDEF_FOUND_FK
ABW_STL_BUILTUP_IBEAM_DEF
ABW_STL_LONG_STIFF
ABW_EVENT_VEHICLE_TEMPLATE
ABW_LEG_ANAL_PT_REINF_CONC
ABW_RESULTS_CRIT_LOAD_LRFD
ABW_RESULTS_DL_ACTION
ABW_RESULTS_LL_ACTION
ABW_RESULTS_LS_SUMMARY_DETAIL
ABW_RESULTS_PS_CONC_STRESS
ABW_RESULTS_RC_SERVICE_STRESS
ABW_RESULTS_SPNG_MBR_ALT_XY
ABW_RESU

In [16]:
import pandas as pd
from sqlalchemy.sql import text

# Query to fetch the entire table
query = text("SELECT * FROM BRIDGEWARE.ABW_GIRDER_MBR")

# Execute the query and fetch the data into a DataFrame
result = oracle_conn.execute(query)
df = pd.DataFrame(result.fetchall(), columns=result.keys())

In [17]:
df

,bridge_id,struct_def_id,super_struct_mbr_id,struct_def_ref_line_id,settlement_load_case_id
0,15974,1,1,5,NaN
1,15974,1,2,6,NaN
2,15974,1,3,7,NaN
3,15974,1,4,8,NaN
4,15974,1,5,9,NaN
...,...,...,...,...,...
73561,17240,1,11,15,NaN
73562,17241,1,1,5,NaN
73563,17241,1,2,6,NaN
73564,17241,1,3,7,NaN


In [18]:
# from civilpy.general import get_table

def get_table_as_df(conn, schema, table):
    query = text(f"SELECT * FROM {schema}.{table}")
    result = conn.execute(query)
    df = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    return df

### BRIDGEWARE

In [19]:
from civilpy.structural.aashtoware.odot_tables import AASHTOWARE_TABLES

In [37]:
# Reloading the variables
# Initialize or clear the variables
all_columns = []
column_lookup = {}
unused_tables = []

# Connect to the SQLite database
conn = sqlite3.connect(r'C:\Users\dparks1\PycharmProjects\civilpy\src\civilpy\data\civilpy.db')
cursor = conn.cursor()

# Query all rows from the metadata table
cursor.execute("SELECT table_name, columns, is_used FROM AASHTOWareMetaData")
rows = cursor.fetchall()

# Rebuild the variables
for table_name, columns_json, is_used in rows:
    columns = json.loads(columns_json)
    all_columns.append(columns)
    column_lookup[table_name] = columns
    if is_used == 0:
        unused_tables.append(table_name)

conn.close()

In [193]:
sfns = get_table("BRIDGEWARE", "ABW_BRIDGE")

# Set the option to display all columns
pd.set_option('display.max_columns', None)

### Lookup BridgeID by SFN

In [41]:
bridge_sfn2 = '0702870'

single_bridge = sfns[sfns['agency_code'] == bridge_sfn2]
bridge_index = single_bridge['bridge_id'].tolist()[0]
bridge_index

13467

### Use the Bridge ID to Find Tables it's in

In [85]:
from IPython.display import display, HTML

data_frames = {}

for key, value in column_lookup.items():
    if 'bridge_id' in value:
        df = get_table("BRIDGEWARE", key)
        df = df[df['bridge_id'] == bridge_index]
        if len(df) > 0:
            print(f"{key} - Stored to dict")
            data_frames[key] = df

ABW_MATL_CONC - Stored to dict
ABW_STL_ANGLE - Stored to dict
ABW_STL_BEAM_ASSEMBLY - Stored to dict
ABW_CONC_RAILING - Stored to dict
ABW_CONC_RAILING_LOC - Stored to dict
ABW_BRIDGE_DIAPHRAGM_DEF_CON - Stored to dict
ABW_SUPER_LOAD_SCENARIO - Stored to dict
ABW_SUPER_LOAD_SCENARIO_ITEM - Stored to dict
ABW_SUPER_STRUCT_LOADING_PATH - Stored to dict
ABW_DIAPH_LOC - Stored to dict
ABW_SUPER_STRUCT_ALT - Stored to dict
ABW_SUPER_STRUCT_MBR - Stored to dict
ABW_BRIDGE_DESIGN_PARAM - Stored to dict
ABW_BRIDGE_DIAPHRAGM_DEF - Stored to dict
ABW_SUPER_LOAD_CASE - Stored to dict
ABW_BEAM_DEF - Stored to dict
ABW_SUPER_STRUCT_DEF_FK - Stored to dict
ABW_SUPER_STRUCT_FK - Stored to dict
ABW_SUPER_STRUCT_SPNG_MBR_FK - Stored to dict
ABW_BRIDGE_DIAPH_DEF_LOC_CON - Stored to dict
ABW_BRIDGE_ENVIRONMENTAL_PARAM - Stored to dict
ABW_LRFD_LS - Stored to dict
ABW_LRFR_FACTOR - Stored to dict
ABW_LRFR_LEGAL_LOADS - Stored to dict
ABW_LRFR_LEGAL_LOADS_HAUL - Stored to dict
ABW_LRFR_LF_TABLE_VALUE - Sto

In [88]:
data_frames['ABW_BRIDGE']

,bridge_id,bridge_guid,agency_code,struct_num,name,bridge_rating_ind,bridge_design_ind,bridge_management_ind,descr,elevation,x_plane_coordinate,y_plane_coordinate,prev_count_year,recent_count_year,recent_count_adtt,prev_adtt,prev_growth_rate,future_count_year,future_count_adtt,future_count_growth_rate,traffic_factor,final_design_event_id,last_modified_event_id,creation_event_id,lrfd_constant_impact_factor,lrfd_fatigue_impact_factor,impact_factor_adjustment,impact_factor_override,impact_factor_type,bridge_units_type,deleted_ind,template_ind,completely_defined_ind,last_change_timestamp,mpf_reduce_based_on_adtt_ind,traffic_adt,traffic_directional_pct,traffic_design_adtt,superstruct_filter_ind,culvert_filter_ind,custom_agency_field_one,custom_agency_field_two,custom_agency_field_three,custom_agency_field_four,custom_agency_field_five,custom_agency_field_six,custom_agency_field_seven,custom_agency_field_eight,custom_agency_field_nine,custom_agency_field_ten,fatigue_importance_factor_type,override_importance_factor_ind,importance_factor_override,sub_struct_filter_ind,exp_annual_adttsl_growth_rate,initial_adttsl,present_adttsl,limit_adttsl,featint,facility,location,yearbuilt,length,routenum,kmpost,adttotal,truckpct,latitude,longitude,district_param_id,county_param_id,funcclass_param_id,nhs_ind_param_id,adminarea_param_id,maintainer_param_id,owner_param_id,bridge_management_guid,bridge_mgmt_sync_event_id,confirm_spatial_ind,sponsoring_agency_codes,member_agency_code,sponsoring_agency_names
10055,13467,339D1250933204FAE0630A10130D66D3,0702870,0702870,BEL-00070-21.340,T,T,F,"Rated by JHL, PE 70217, from Original 1979 Pla...",NaN,0.0,0.0,None,None,NaN,None,None,None,None,None,None,None,NaN,1120866.0,33.0,15.0,NaN,NaN,31401,10401,None,F,F,2021-08-18 20:13:17,None,20981.0,NaN,NaN,T,F,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,T,NaN,NaN,NaN,NaN,IR 470 (WB Lanes Only),IR 70,At Juction with IR 470,1982.0,72.29856,70,34.343401,NaN,NaN,40.07,-80.84,135.0,15.0,142.0,156.0,3.0,99.0,99.0,None,None,None,None,NaN,None


In [ ]:
# Useful columns repeated elsewhere
# longitude
# latitude
# district_param_id - Needs Definition
# county_param_id - Needs Definition
# funcclast_param_id - Needs Definiton
# nhs_ind_param_id	
# adminarea_param_id	
# maintainer_param_id	
# owner_param_id
# featint
# facility
# location
# yearbuilt
# length
# routenum
# kmpost  # Would like background on this...
# adttotal
# truckpct

In [94]:
sfns['county_param_id'].unique()

array([nan, 69., 71., 78., 64., 60., 84., 20., 21., 79.,  9., 95.,  7.,
       33., 80., 76., 12., 74., 65., 26., 11., 32., 28., 39., 75., 67.,
       73., 29., 58., 88., 23., 85., 93., 16., 96., 25., 42., 51., 62.,
       56., 68., 86., 10., 55., 40., 35., 27., 83., 18., 52., 14., 54.,
       22., 87., 17., 53., 41., 63., 91., 15., 50., 31., 45., 44., 36.,
       89., 19., 13., 49., 48., 46., 37., 77., 81., 24., 43., 38., 61.,
       30., 90., 92., 82., 94., 72., 47., 34., 70.,  8., 57., 66., 59.])

In [99]:
for i in range(0, 101, 1):
    if i not in sfns['county_param_id'].unique():
        print(i)

0
1
2
3
4
5
6
97
98
99
100


In [100]:
from sqlalchemy import text

def find_foreign_key_reference(table_name, column_name, schema_name, connection):
    """
    Finds the referenced table and column for a given foreign key column.
    
    Parameters:
        table_name (str): The name of the table containing the foreign key.
        column_name (str): The foreign key column name.
        schema_name (str): The schema that owns the table.
        connection (sqlalchemy.engine.Connection): Active SQLAlchemy connection.
    
    Returns:
        DataFrame with referenced table and column info.
    """
    query = text("""
    SELECT
        acc.table_name AS source_table,
        acc.column_name AS source_column,
        c_pk.table_name AS referenced_table,
        acc_r.column_name AS referenced_column,
        ac.constraint_name
    FROM
        all_constraints ac
        JOIN all_cons_columns acc ON ac.owner = acc.owner AND ac.constraint_name = acc.constraint_name
        JOIN all_constraints c_pk ON ac.r_owner = c_pk.owner AND ac.r_constraint_name = c_pk.constraint_name
        JOIN all_cons_columns acc_r ON c_pk.owner = acc_r.owner AND c_pk.constraint_name = acc_r.constraint_name AND acc.position = acc_r.position
    WHERE
        ac.constraint_type = 'R'
        AND acc.table_name = :table_name
        AND acc.column_name = :column_name
        AND ac.owner = :schema_name
    """)

    return pd.read_sql(query, connection, params={
        "table_name": table_name.upper(),
        "column_name": column_name.upper(),
        "schema_name": schema_name.upper()
    })


In [101]:
df_fk = find_foreign_key_reference(
    table_name='ABW_BRIDGE',
    column_name='county_param_id',
    schema_name='BRIDGEWARE',  # Replace with your actual schema
    connection=oracle_conn
)

In [102]:
df_fk

,source_table,source_column,referenced_table,referenced_column,constraint_name
0,ABW_BRIDGE,COUNTY_PARAM_ID,ABW_PARAMETER,PARAMETER_ID,R_5470


In [104]:
df = get_table("BRIDGEWARE", "ABW_PARAMETER")

In [108]:
bridge_values = df[df['table_name'] == 'bridge']

In [181]:
df = bridge_values

In [182]:
df

,parameter_id,table_name,field_name,parmvalue,shortdesc,longdesc
0,160,bridge,adminarea,5,Private,None
1,161,bridge,adminarea,6,Border State,None
2,162,bridge,adminarea,7,Other,None
3,1,bridge,adminarea,-1,Unknown,Unknown
4,2,bridge,adminarea,-2,Not Applicable,Not Applicable
...,...,...,...,...,...,...
134,132,bridge,district,08,08,None
135,133,bridge,district,09,09,None
136,134,bridge,district,10,10,None
137,135,bridge,district,11,11,None


In [194]:
# Create a mapping dictionary from df
param_lookup = df.set_index('parameter_id')['shortdesc'].to_dict()

# Apply the mapping to create a new column in sfns
sfns['district_param_id'] = sfns['district_param_id'].dropna().astype(int).map(param_lookup)
sfns['county_param_id'] = sfns['county_param_id'].dropna().astype(int).map(param_lookup)
sfns['funcclass_param_id'] = sfns['funcclass_param_id'].dropna().astype(int).map(param_lookup)
sfns['nhs_ind_param_id'] = sfns['nhs_ind_param_id'].dropna().astype(int).map(param_lookup)
sfns['adminarea_param_id'] = sfns['adminarea_param_id'].dropna().astype(int).map(param_lookup)
sfns['maintainer_param_id'] = sfns['maintainer_param_id'].dropna().astype(int).map(param_lookup)
sfns['owner_param_id'] = sfns['owner_param_id'].dropna().astype(int).map(param_lookup)

In [196]:
sfns[sfns['agency_code'] == '0702870']

,bridge_id,bridge_guid,agency_code,struct_num,name,bridge_rating_ind,bridge_design_ind,bridge_management_ind,descr,elevation,x_plane_coordinate,y_plane_coordinate,prev_count_year,recent_count_year,recent_count_adtt,prev_adtt,prev_growth_rate,future_count_year,future_count_adtt,future_count_growth_rate,traffic_factor,final_design_event_id,last_modified_event_id,creation_event_id,lrfd_constant_impact_factor,lrfd_fatigue_impact_factor,impact_factor_adjustment,impact_factor_override,impact_factor_type,bridge_units_type,deleted_ind,template_ind,completely_defined_ind,last_change_timestamp,mpf_reduce_based_on_adtt_ind,traffic_adt,traffic_directional_pct,traffic_design_adtt,superstruct_filter_ind,culvert_filter_ind,custom_agency_field_one,custom_agency_field_two,custom_agency_field_three,custom_agency_field_four,custom_agency_field_five,custom_agency_field_six,custom_agency_field_seven,custom_agency_field_eight,custom_agency_field_nine,custom_agency_field_ten,fatigue_importance_factor_type,override_importance_factor_ind,importance_factor_override,sub_struct_filter_ind,exp_annual_adttsl_growth_rate,initial_adttsl,present_adttsl,limit_adttsl,featint,facility,location,yearbuilt,length,routenum,kmpost,adttotal,truckpct,latitude,longitude,district_param_id,county_param_id,funcclass_param_id,nhs_ind_param_id,adminarea_param_id,maintainer_param_id,owner_param_id,bridge_management_guid,bridge_mgmt_sync_event_id,confirm_spatial_ind,sponsoring_agency_codes,member_agency_code,sponsoring_agency_names
10055,13467,339D1250933204FAE0630A10130D66D3,0702870,0702870,BEL-00070-21.340,T,T,F,"Rated by JHL, PE 70217, from Original 1979 Pla...",NaN,0.0,0.0,None,None,NaN,None,None,None,None,None,None,None,NaN,1120866.0,33.0,15.0,NaN,NaN,31401,10401,None,F,F,2021-08-18 20:13:17,None,20981.0,NaN,NaN,T,F,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,T,NaN,NaN,NaN,NaN,IR 470 (WB Lanes Only),IR 70,At Juction with IR 470,1982.0,72.29856,70,34.343401,NaN,NaN,40.07,-80.84,11,Belmont,NaN,NaN,ODOT,01 State Highway Agency,01 State Highway Agency,None,None,None,None,NaN,None


,bridge_id,bridge_guid,agency_code,struct_num,name,bridge_rating_ind,bridge_design_ind,bridge_management_ind,descr,elevation,x_plane_coordinate,y_plane_coordinate,prev_count_year,recent_count_year,recent_count_adtt,prev_adtt,prev_growth_rate,future_count_year,future_count_adtt,future_count_growth_rate,traffic_factor,final_design_event_id,last_modified_event_id,creation_event_id,lrfd_constant_impact_factor,lrfd_fatigue_impact_factor,impact_factor_adjustment,impact_factor_override,impact_factor_type,bridge_units_type,deleted_ind,template_ind,completely_defined_ind,last_change_timestamp,mpf_reduce_based_on_adtt_ind,traffic_adt,traffic_directional_pct,traffic_design_adtt,superstruct_filter_ind,culvert_filter_ind,custom_agency_field_one,custom_agency_field_two,custom_agency_field_three,custom_agency_field_four,custom_agency_field_five,custom_agency_field_six,custom_agency_field_seven,custom_agency_field_eight,custom_agency_field_nine,custom_agency_field_ten,fatigue_importance_factor_type,override_importance_factor_ind,importance_factor_override,sub_struct_filter_ind,exp_annual_adttsl_growth_rate,initial_adttsl,present_adttsl,limit_adttsl,featint,facility,location,yearbuilt,length,routenum,kmpost,adttotal,truckpct,latitude,longitude,district_param_id,county_param_id,funcclass_param_id,nhs_ind_param_id,adminarea_param_id,maintainer_param_id,owner_param_id,bridge_management_guid,bridge_mgmt_sync_event_id,confirm_spatial_ind,sponsoring_agency_codes,member_agency_code,sponsoring_agency_names,district_longdesc
0,19483,339D12506E8C04FAE0630A10130D66D3,PSBD-1-25 (B42-48) - 95',B42-48 - 95' span,B42-48 - 95' span N/G,T,T,F,FAILED LOAD RATING/ DESIGN. DO NOT USE\r\nPres...,NaN,NaN,NaN,None,None,NaN,None,None,None,None,None,None,None,1204822.0,1194654.0,33.0,15.0,NaN,NaN,31401,10401,None,F,T,2025-06-16 18:19:55.857281,F,NaN,NaN,NaN,T,F,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,F,NaN,NaN,NaN,NaN,None,None,None,NaN,28.95600,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,F,None,1541.0,None,NaN
1,18590,339D12506E8D04FAE0630A10130D66D3,PSBD-1-25 (CB42-48) - 90',CB42-48 - 90' span,CB42-48 - 90' span,T,T,F,"Prestressed concrete box bridge, 90' single sp...",NaN,NaN,NaN,None,None,NaN,None,None,None,None,None,None,None,1204843.0,1194654.0,33.0,15.0,NaN,NaN,31401,10401,None,F,T,2025-06-16 19:12:05.827640,F,NaN,NaN,NaN,T,F,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,F,NaN,NaN,NaN,NaN,None,None,None,NaN,27.43200,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,F,None,1541.0,None,NaN
2,18873,339D12506E8E04FAE0630A10130D66D3,6104487,6104487,NOB-564-7.730,T,T,F,"Rated by RTF, PE 80163 using existing plans (1...",NaN,NaN,NaN,None,None,NaN,None,None,None,None,None,None,None,1196744.0,1196743.0,33.0,15.0,NaN,NaN,31401,10401,None,F,F,2024-11-06 17:51:59.970683,None,NaN,NaN,NaN,T,F,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,F,NaN,NaN,NaN,NaN,Unnamed Stream,SR 564,9.3 Km W Of Jct SR 145,1997.0,12.92352,564,12.459541,NaN,NaN,39.67,-81.40,10,69.0,146.0,155.0,3.0,99.0,99.0,None,None,F,None,NaN,None,10
3,18874,339D12506E8F04FAE0630A10130D66D3,6300162,6300162,PAU-24-04.600,T,T,F,"Rated by RTF, PE 80163 using existing plans (2...",NaN,NaN,NaN,None,None,NaN,None,None,None,None,None,None,None,1196746.0,1196745.0,33.0,15.0,NaN,NaN,31401,10401,None,F,F,2024-11-06 17:52:01.149018,None,NaN,NaN,NaN,F,T,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,F,NaN,NaN,NaN,NaN,NORTH CREEK,USR 24,US-24 Over North Creek,2009.0,6.70560,24,7.394936,NaN,NaN,41.17,-84.72,01,71.0,143.0,156.0,3.0,99.0,99.0,None,None,F,None,NaN,None,01
4,96,339D12506E9004FAE0630A10130D66D3,7004133,7004133,RIC-71-1068 R,T,T,F,"Rating by AP, from plans and standards, Septem...",NaN,0.0,0.0,None,None,0.0,None,None,None,None,None,None,None,1203852.0,5386.0,NaN,NaN,NaN,NaN,31401,10401,None,F,F,2025-05-28 19:46:36.956179,None,NaN,NaN,NaN,T,F,None,None,None,None,None,None,None,None,None,None,59301,F,NaN,T,NaN,NaN,NaN,NaN,S

In [56]:
df_fks

,constraint_name,table_name,column_name,referenced_table,referenced_column


In [51]:
data_frames['ABW_GIRDER_SYS_STRUCT_DEF']

,bridge_id,struct_def_id,dl_distribution2_type,dl_distribution1_type,modular_ratio_sustained_factor,deck_crack_control_param_z,girder_spacing_display_type,wearing_surface_matl_name,wearing_surface_desc,wearing_surface_density,wearing_surface_thick,mbr_type,deck_type,frame_struct_simple_def_ind,truck_traffic_frac_single_lane,num_lanes_avail_truck,override_truck_traffic_ind,deck_exposure_factor,thick_field_mea_ind,dist_left_girder_stdef_refline,modeling_type,frame_struct_simp_def_ind,sacrificial_wear_thick,struct_overlay_thick,struct_overlay_density,sustain_modular_ratio,top_slab_crack_ctrl_param,top_slab_exposure_fact,stagger_perpend_diaphragms_ind
3400,13467,1,33301,33201,3.0,NaN,27602,WS,None,2322.716258,88.9,NaN,36201,None,None,4.0,F,NaN,T,NaN,61301,None,None,None,None,None,None,None,None
3401,13467,2,33301,33201,3.0,NaN,27602,WS,None,2322.716258,88.9,NaN,36201,None,None,3.0,F,NaN,T,NaN,61301,None,None,None,None,None,None,None,None


In [50]:
data_frames['ABW_SUPER_STRUCT_MBR_SPAN']

,bridge_id,struct_def_id,super_struct_mbr_id,span_id,dist,length
91050,13467,1,1,1,0.0000,23.0124
91051,13467,1,1,2,23.0124,28.4988
91052,13467,1,1,3,51.5112,18.7452
91053,13467,1,2,1,0.0000,23.0124
91054,13467,1,2,2,23.0124,28.4988
91055,13467,1,2,3,51.5112,18.7452
91056,13467,1,3,1,0.0000,23.0124
91057,13467,1,3,2,23.0124,28.4988
91058,13467,1,3,3,51.5112,18.7452
91059,13467,1,4,1,0.0000,23.0124


# Assetwise

# PlanVault ESRI Endpoint

https://collectorweb.dot.state.oh.us/arcgis/rest/services/CADD_PlanVault/CADD_PlanVault_Prod/FeatureServer/0

https://gis3.dot.state.oh.us/cadd_planvault/

In [16]:
import requests
import pandas as pd

# The URL for the query operation.
query_url = "https://collectorweb.dot.state.oh.us/arcgis/rest/services/CADD_PlanVault/CADD_PlanVault_Prod/FeatureServer/0/query"

# Define the base query parameters
params = {
    "f": "json",
    "where": "1=1",
    "outFields": "GUID,OBJECTID,NLFID,CRS,PID,PRIMARY_WORK_CATEGORY,PLAN_DATE,PLAN_URL,SFN",
    "returnGeometry": "false",
    "resultRecordCount": 1000  # Number of records to request per query
}

# List to hold all the retrieved records
all_records = []
offset = 0

print("Starting data retrieval...")

try:
    while True:
        # Update the offset for the current request
        params["resultOffset"] = offset
        
        # Make the HTTP GET request
        response = requests.get(query_url, params=params)
        response.raise_for_status()  # Check for HTTP errors
        
        data = response.json()
        
        if "error" in data:
            print(f"Server error: {data['error']['message']}")
            break
            
        if "features" in data and data["features"]:
            # Extract attributes from each feature
            records = [feature["attributes"] for feature in data["features"]]
            all_records.extend(records)
            
            # Update the offset for the next iteration
            records_count = len(records)
            offset += records_count
            print(f"Retrieved {records_count} records. Total so far: {offset}")
            
            # If the number of records retrieved is less than the requested amount, we've reached the end.
            if records_count < params["resultRecordCount"]:
                print("End of dataset reached.")
                break
        else:
            print("No more features to retrieve.")
            break

    # Create the DataFrame once all records are collected
    if all_records:
        df = pd.DataFrame(all_records)
        print("\nDataFrame successfully created.")
        print(df.head())
        print(f"\nTotal records retrieved: {len(df)}")
    else:
        print("Failed to retrieve any data.")

except requests.exceptions.RequestException as e:
    print(f"An error occurred during the request: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Starting data retrieval...
Retrieved 1000 records. Total so far: 1000
Retrieved 1000 records. Total so far: 2000
Retrieved 1000 records. Total so far: 3000
Retrieved 1000 records. Total so far: 4000
Retrieved 1000 records. Total so far: 5000
Retrieved 1000 records. Total so far: 6000
Retrieved 1000 records. Total so far: 7000
Retrieved 1000 records. Total so far: 8000
Retrieved 1000 records. Total so far: 9000
Retrieved 1000 records. Total so far: 10000
Retrieved 1000 records. Total so far: 11000
Retrieved 1000 records. Total so far: 12000
Retrieved 1000 records. Total so far: 13000
Retrieved 1000 records. Total so far: 14000
Retrieved 1000 records. Total so far: 15000
Retrieved 1000 records. Total so far: 16000
Retrieved 1000 records. Total so far: 17000
Retrieved 1000 records. Total so far: 18000
Retrieved 1000 records. Total so far: 19000
Retrieved 1000 records. Total so far: 20000
Retrieved 1000 records. Total so far: 21000
Retrieved 1000 records. Total so far: 22000
Retrieved 1000

In [10]:
df['PLAN_URL'].iloc[1]

'\\\\filestore\\CaddMappingSurvey\\ProjectWise\\FinalPlanSets\\PID\\1998\\MUS-2037'

In [75]:
def query_feature_service():
    """
    Queries the feature service with pagination to get all records for the
    specified fields and returns them in a pandas DataFrame.
    """
    all_records = []
    offset = 0
    TARGET_FIELDS = "PID,SFN,GUID,CRS"
    
    print("Step 1: Querying the feature service to get all records...")
    
    try:
        while True:
            params = {
                "f": "json",
                "outFields": TARGET_FIELDS,
                "where": "1=1",
                "returnGeometry": "false",
                "resultRecordCount": RECORDS_PER_PAGE,
                "resultOffset": offset
            }
            
            response = requests.get(FEATURE_SERVICE_URL, params=params)
            response.raise_for_status()
            
            data = response.json()
            
            if "error" in data:
                print(f"Server error: {data['error']['message']}")
                return None
            
            records_retrieved = len(data.get("features", []))
            print(f"    - Retrieved {records_retrieved} records with offset {offset}.")
            
            if records_retrieved > 0:
                records = [feature["attributes"] for feature in data["features"]]
                all_records.extend(records)
                offset += records_retrieved
            
            if not data.get("exceededTransferLimit", False):
                break

    except requests.exceptions.RequestException as e:
        print(f"Error querying feature service: {e}")
        return None
        
    if all_records:
        df = pd.DataFrame(all_records)
        print(f"Successfully retrieved a DataFrame with {len(df)} total records.")
        return df
    else:
        print("Failed to retrieve any data from the feature service.")
        return None

In [105]:
def get_pdf_url_from_guid(guid):
    """
    Submits a geoprocessing job for a single GUID, polls for the result,
    and returns the final PDF URL.
    """
    submit_url = urljoin(GEOPROCESSING_BASE_URL, "submitJob")
    submit_params = {"f": "json", "planguid": guid}
    
    # Add headers to mimic a browser request from the official site
    # This is crucial for cross-origin resource sharing (CORS) validation
    headers = {
        "Referer": "https://gis3.dot.state.oh.us/",
        "Origin": "https://gis3.dot.state.oh.us"
    }
    
    print(f"    - Submitting job for GUID: {guid}")
    print(f"    - Requesting URL: {submit_url} with data: {submit_params}")
    
    job_id = None
    for attempt in range(MAX_JOB_SUBMIT_RETRIES):
        try:
            # Pass the headers dictionary in the POST request
            response = requests.post(submit_url, data=submit_params, headers=headers, timeout=10)
            response.raise_for_status()
            job_data = response.json()
            job_id = job_data.get("jobId")
            if job_id:
                break
        except requests.exceptions.RequestException as e:
            print(f"    - Failed to submit job on attempt {attempt + 1}: {e}")
            time.sleep(5)
    
    if not job_id:
        print(f"    - Failed to submit job for GUID: {guid} after {MAX_JOB_SUBMIT_RETRIES} attempts.")
        return None

    # Assume job_id is successfully retrieved from the job submission step
    job_status_url = f"https://collectornew.dot.state.oh.us/arcgis/rest/services/CADD_PlanVault/ProjectWisePlanSetReaderByGuid/GPServer/PlanVault_ProjectWisePlanSetReaderByGuid/jobs/{job_id}"
    results_url = f"{job_status_url}/results/planSet"
    
    try:
        results_params = {"f": "json"}
        print(f"    - Requesting results URL: {results_url} with parameters: {results_params}")

        # Step 2: Request the JSON with the PDF URL
        results_response = requests.get(results_url, params=results_params, headers=headers, timeout=10)
        results_response.raise_for_status()  # This will raise an exception for a bad status code
        results_data = results_response.json()
        
        print(f"    - Received JSON response: {results_data}")

        # Extract the PDF URL from the JSON
        pdf_download_url = results_data.get("value", {}).get("url")

        if pdf_download_url:
            print(f"    - PDF download URL found: {pdf_download_url}")
            
            # Step 3: Use the new URL to download the PDF
            print(f"    - Downloading PDF from URL: {pdf_download_url}")
            
            # The headers may not be strictly required for the final download, but it's safe to include them.
            pdf_response = requests.get(pdf_download_url, headers=headers, stream=True)
            pdf_response.raise_for_status()
            
            # Save the file to disk
            filename = pdf_download_url.split('/')[-1]
            with open(filename, 'wb') as f:
                for chunk in pdf_response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"    - Successfully downloaded and saved: {filename}")
            return filename
        else:
            print("    - 'url' key not found in the 'value' object.")
            return None

    except requests.exceptions.RequestException as e:
        print(f"    - Error during results retrieval or PDF download: {e}")
        return None

In [87]:
output_dir = r"C:\Users\dparks1\OneDrive - State of Ohio\Documents\DownloadedPlans"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Get the initial DataFrame with all records.
plans_df = query_feature_service()

# Filter for rows with a valid SFN.
plans_df['SFN'] = plans_df['SFN'].replace('', np.nan)
filtered_df = plans_df.dropna(subset=['SFN']).reset_index(drop=True)

Step 1: Querying the feature service to get all records...
    - Retrieved 1000 records with offset 0.
    - Retrieved 1000 records with offset 1000.
    - Retrieved 1000 records with offset 2000.
    - Retrieved 1000 records with offset 3000.
    - Retrieved 1000 records with offset 4000.
    - Retrieved 1000 records with offset 5000.
    - Retrieved 1000 records with offset 6000.
    - Retrieved 1000 records with offset 7000.
    - Retrieved 1000 records with offset 8000.
    - Retrieved 1000 records with offset 9000.
    - Retrieved 1000 records with offset 10000.
    - Retrieved 1000 records with offset 11000.
    - Retrieved 1000 records with offset 12000.
    - Retrieved 1000 records with offset 13000.
    - Retrieved 1000 records with offset 14000.
    - Retrieved 1000 records with offset 15000.
    - Retrieved 1000 records with offset 16000.
    - Retrieved 1000 records with offset 17000.
    - Retrieved 1000 records with offset 18000.
    - Retrieved 1000 records with offset 1

In [106]:
def download_pdf(url, output_path, sfn):
    """
    Downloads a PDF from a URL and saves it to the specified path.
    """
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        
        file_name = f"{sfn} - {os.path.basename(url)}"
        full_path = os.path.join(output_path, file_name)
        
        with open(full_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print(f"    - PDF successfully downloaded to {full_path}")
        return True
    except requests.exceptions.RequestException as e:
        print(f"    - Error downloading the PDF: {e}")
        return False

In [111]:
import requests
from pathlib import Path
import time
from urllib.parse import urljoin

# Headers to mimic a browser, which are necessary for this service
headers = {
    "Referer": "https://gis3.dot.state.oh.us/",
    "Origin": "https://gis3.dot.state.oh.us"
}

output_dir = r"C:\Users\dparks1\OneDrive - State of Ohio\Documents\DownloadedPlans"

# Define base URL and timeouts
GEOPROCESSING_BASE_URL = "https://collectornew.dot.state.oh.us/arcgis/rest/services/CADD_PlanVault/ProjectWisePlanSetReaderByGuid/GPServer/PlanVault_ProjectWisePlanSetReaderByGuid/"
JOB_TIMEOUT_SECONDS = 300
POLL_INTERVAL_SECONDS = 5

def download_plan(guid):
    """
    Submits a geoprocessing job, polls for a successful status,
    retrieves the final JSON, and downloads the PDF file.
    """
    submit_url = urljoin(GEOPROCESSING_BASE_URL, "submitJob")
    submit_params = {"f": "json", "planguid": guid}
    
    print(f"    - Submitting job for GUID: {guid}")
    print(f"    - Requesting URL: {submit_url} with data: {submit_params}")
    
    # Step 1: Submit the job
    try:
        response = requests.post(submit_url, data=submit_params, headers=headers)
        response.raise_for_status()
        job_data = response.json()
        job_id = job_data.get("jobId")

        if not job_id:
            print("    - Failed to get a jobId from the server.")
            return None
    except requests.exceptions.RequestException as e:
        print(f"    - Error submitting job: {e}")
        return None

    # Step 2: Poll for job status
    job_status_url = urljoin(GEOPROCESSING_BASE_URL, f"jobs/{job_id}")
    start_time = time.time()

    while True:
        try:
            print(f"    - Polling status URL: {job_status_url} with parameters: {{'f': 'json'}}")
            status_response = requests.get(job_status_url, params={"f": "json"}, headers=headers, timeout=10)
            status_response.raise_for_status()
            status_data = status_response.json()
            job_status = status_data.get("jobStatus")
            
            if job_status == "esriJobSucceeded":
                print("    - Job succeeded. Proceeding to download.")
                break
            elif job_status in ["esriJobFailed", "esriJobCancelled"]:
                print(f"    - Job for GUID {guid} failed with status: {job_status}.")
                return None
            
            if time.time() - start_time > JOB_TIMEOUT_SECONDS:
                print(f"    - Job for GUID {guid} timed out after {JOB_TIMEOUT_SECONDS} seconds.")
                return None
            
            time.sleep(POLL_INTERVAL_SECONDS)

        except requests.exceptions.RequestException as e:
            print(f"    - Error while polling job status: {e}")
            return None
    
    # Step 3: Retrieve the results
    results_url = f"{job_status_url}/results/planSet"
    results_params = {"f": "json"}

    try:
        print(f"    - Requesting results URL: {results_url} with parameters: {results_params}")
        results_response = requests.get(results_url, params=results_params, headers=headers, timeout=10)
        results_response.raise_for_status()
        results_data = results_response.json()

        print(f"    - Received JSON response: {results_data}")
        pdf_download_url = results_data.get("value", {}).get("url")

        if pdf_download_url:
            print(f"    - PDF download URL found: {pdf_download_url}")
            
            # Final step: Download the PDF from the URL provided in the JSON
            pdf_response = requests.get(pdf_download_url, headers=headers, stream=True)
            pdf_response.raise_for_status()
            
            filename = pdf_download_url.split('/')[-1]
            with open(Path(output_dir) / filename, 'wb') as f:
                for chunk in pdf_response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"    - Successfully downloaded and saved: {filename}")
            return filename
        else:
            print("    - 'url' key not found in the 'value' object.")
            return None

    except requests.exceptions.RequestException as e:
        print(f"    - Error during results retrieval or PDF download: {e}")
        return None

In [121]:
guids = list(filtered_df['GUID'].unique())

In [123]:
for i in range(67, len(guids)):
    print(f"\n{i}")
    download_plan(guids[i])


67
    - Submitting job for GUID: 48dd5d11-dab7-4798-a67a-9034b4edb1af
    - Requesting URL: https://collectornew.dot.state.oh.us/arcgis/rest/services/CADD_PlanVault/ProjectWisePlanSetReaderByGuid/GPServer/PlanVault_ProjectWisePlanSetReaderByGuid/submitJob with data: {'f': 'json', 'planguid': '48dd5d11-dab7-4798-a67a-9034b4edb1af'}
    - Polling status URL: https://collectornew.dot.state.oh.us/arcgis/rest/services/CADD_PlanVault/ProjectWisePlanSetReaderByGuid/GPServer/PlanVault_ProjectWisePlanSetReaderByGuid/jobs/je603690709094c579f93443aa155de27 with parameters: {'f': 'json'}
    - Polling status URL: https://collectornew.dot.state.oh.us/arcgis/rest/services/CADD_PlanVault/ProjectWisePlanSetReaderByGuid/GPServer/PlanVault_ProjectWisePlanSetReaderByGuid/jobs/je603690709094c579f93443aa155de27 with parameters: {'f': 'json'}
    - Polling status URL: https://collectornew.dot.state.oh.us/arcgis/rest/services/CADD_PlanVault/ProjectWisePlanSetReaderByGuid/GPServer/PlanVault_ProjectWisePlanS

KeyboardInterrupt: 

## Appendix